# Funding agency classification with GPT-5 mini (version 2)

The manuscript classifies funding organizations into four categories and identifies whether each organization is U.S.-based. This notebook documents the prompts and classification procedure used for the second version. It requires an institutionally authorized input file containing agency names and aliases. That file is not distributed. No API calls have been made in preparing this notebook.


## Inputs and credentials

The restricted input must contain `original_agency`, `preferred_name`, and `count`. The `count` field is carried to the output but is not included in either classification prompt. Configure `OPENAI_API_KEY` in the execution environment; do not place a key in a notebook.


In [ ]:
import csv
import json
import os
from pathlib import Path

from openai import OpenAI

INPUT_PATH = Path("restricted_agency_input.csv")
OUTPUT_PATH = Path("agency_classification_eq5_v2.jsonl")
MODEL = "gpt-5-mini"

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the authorized environment before running.")

client = OpenAI()


## Classification prompts

The first prompt assigns agency type. The second assigns U.S. or non-U.S. status. Both use the original agency name and any available aliases.


In [ ]:
SYSTEM_PROMPT_V2 = """
You are a classification assistant.

Context:
The institution names come from funding agencies extracted from U.S. medical-related academic papers.

Task:
Classify each institution into exactly one of the following categories:
1) Public Agency (government-affiliated; exclude universities)
2) Private Company
3) University
4) Other Organization

Rules:
- University-affiliated institutions (including affiliated hospitals) must be classified as University.
- Institutions explicitly identified as non-profit but not directly affiliated with government must be classified as Other Organization.
- Output only the category name exactly as listed above, nothing else.
- Do not include explanations.
- If uncertain, choose the closest category.
"""

SYSTEM_PROMPT_COUNTRY_V2 = """
You are a classification assistant.

Context:
The institution names come from funding agencies extracted from U.S. medical-related literature.
Because of this context, institutions with names like "National Institutes of Health" or "National Science Foundation (NSF)" are very likely to be U.S.-based institutions.

Task:
Decide the country classification for each institution.

Categories:
- "US" (United States)
- "Non-US" (outside the United States, including multinational/global organizations such as the United Nations or WHO)
- "Uncertain" (only if the name is highly ambiguous and you truly cannot determine)

Rules:
- For companies that were acquired, classify based on the country of origin before acquisition.
- For joint ventures between the U.S. and other countries, classify as "US".
- For multinational or intergovernmental organizations (e.g., UN, WHO), classify as "Non-US".
- Output only one of: "US", "Non-US", or "Uncertain".
- Do not include explanations.
- Be decisive; use "Uncertain" only if absolutely necessary.
"""


## Classify one organization

The type response is mapped to one of the four permitted labels. The original workflow assigned unmatched type responses to `Other Organization` and unmatched country responses to `Uncertain`; those fallback rules are retained here.


In [ ]:
def classify_agency(original_name, aliases, client=client):
    if isinstance(aliases, (list, tuple, set)):
        aliases = ", ".join(map(str, aliases))
    else:
        aliases = str(aliases)

    user_msg = (
        "Classify the funding agency using the main name and optional aliases.\n"
        f"Main name: {original_name}\n"
        f"Aliases: {aliases}\n"
        "Answer with exactly one of: Public Agency; Private Company; University; Other Organization."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_V2},
            {"role": "user", "content": user_msg},
        ],
        reasoning_effort="minimal",
    )
    raw = (response.choices[0].message.content or "").strip().lower()
    if "public agency" in raw:
        category = "Public Agency"
    elif "private company" in raw:
        category = "Private Company"
    elif "university" in raw:
        category = "University"
    elif "other organization" in raw:
        category = "Other Organization"
    else:
        category = "Other Organization"

    country_msg = (
        "Decide whether the institution is based in the United States using the main name and optional aliases.\n"
        f"Main name: {original_name}\n"
        f"Aliases: {aliases}\n"
        "Answer with exactly one of: US; Non-US; Uncertain."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_COUNTRY_V2},
            {"role": "user", "content": country_msg},
        ],
        reasoning_effort="minimal",
    )
    raw_country = (response.choices[0].message.content or "").strip().upper()
    if raw_country == "US":
        country = "US"
    elif raw_country == "NON-US":
        country = "Non-US"
    elif raw_country == "UNCERTAIN":
        country = "Uncertain"
    else:
        country = "Uncertain"
    return category, country


## Process the restricted input

The original classification covered agencies appearing at least five times. The loop below records each classification as JSON Lines. It is provided for review and must only be run where use of the licensed input is authorized.


In [ ]:
def classify_file(input_path=INPUT_PATH, output_path=OUTPUT_PATH):
    with input_path.open(encoding="utf-8-sig", newline="") as source, output_path.open(
        "w", encoding="utf-8"
    ) as destination:
        for row in csv.DictReader(source):
            category, country = classify_agency(
                row["original_agency"], row["preferred_name"]
            )
            result = {
                "original_agency": row["original_agency"],
                "preferred_name": row["preferred_name"],
                "count": row["count"],
                "category": category,
                "country": country,
            }
            destination.write(json.dumps(result, ensure_ascii=False) + "\n")

# In an authorized environment, call classify_file() after supplying the restricted input.


## Manual validation

The accompanying 1,000-row CSV compares GPT-5 mini v2 labels with independent manual labels. The public CSV uses sequential sample IDs and omits agency names, aliases, and frequencies. The manuscript reports 92.1% agreement for agency type.


In [ ]:
def summarize_validation(path="agency_classification_validation_1000.csv"):
    with open(path, encoding="utf-8", newline="") as source:
        rows = list(csv.DictReader(source))
    category_agreement = sum(
        row["gpt5_v2_category"] == row["manual_category"] for row in rows
    ) / len(rows)
    country_agreement = sum(
        row["gpt5_v2_country"] == row["manual_country"] for row in rows
    ) / len(rows)
    return {
        "sample_size": len(rows),
        "category_agreement": category_agreement,
        "country_agreement": country_agreement,
    }

# Call summarize_validation() to inspect the published label comparison.
